# 02 · Cleaning (Pandas)

Runs `src/clean.py`. Problems are **flagged rather than dropped** so the official benchmark split stays intact. Every decision, with row counts, is written to `reports/cleaning_log.md`.

Steps that the brief asked for but are impossible in this release (epoch conversion of Stime/Ltime, sport/dsport coercion, hour/day features) are logged as such.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
SRC = Path.cwd().parent / 'src' if (Path.cwd().parent / 'src').exists() else Path.cwd() / 'src'
sys.path.insert(0, str(SRC))
import pandas as pd, json
from IPython.display import Image, Markdown, display
from config import FIG_DIR, METRICS_DIR, REPORTS_DIR, EXPORT_DIR
pd.set_option('display.max_columns', 60, 'display.width', 200)
def show(name, width=900):
    display(Image(filename=str(FIG_DIR / f'{name}.png'), width=width))

In [2]:
import clean
summary = clean.run()
summary


PHASE 3 - CLEANING


 - Column names: Header row present; names stripped and lower-cased (no NUSW feature-file mapping needed for this release) (affected 0; rows 257,673 -> 257,673)
 - Unavailable fields: srcip, dstip, sport, dsport, stime, ltime do NOT exist in the train/test release -> no epoch conversion, port coercion, hour/day features or IP analysis. Substitutes are documented in the README (affected 0; rows 257,673 -> 257,673)
 - Categorical text: Stripped whitespace; proto/service lower-case, state upper-case (affected 1; rows 257,673 -> 257,673)
 - Service '-': '-' means no application service was identified -> recoded to 'none' (affected 141,321; rows 257,673 -> 257,673)
 - attack_cat missing: NaN/blank attack_cat filled with 'Normal' (affected 0; rows 257,673 -> 257,673)
 - attack_cat labels: Stripped + mapped spelling variants (e.g. 'Backdoors'->'Backdoor') (affected 0; rows 257,673 -> 257,673)
 - Label consistency: Checked label==0 <=> attack_cat=='Normal'; mismatches flagged (flag_label_misma

 - Duplicates (within split): Feature-identical rows (ignoring id) FLAGGED, not removed: dropping them would change the official benchmark split, and repeated identical flows are genuine traffic (e.g. scripted Generic/Exploit attacks) (affected 93,824; rows 257,673 -> 257,673)


 - Train/test overlap: Test rows whose features exactly match a train row flagged (in_both_splits) - reported as an evaluation caveat (affected 4,247; rows 257,673 -> 257,673)
 - Derived features: Added is_attack, total_bytes, total_pkts, bytes_per_sec, pkts_per_sec, byte_ratio=sbytes/(dbytes+1), src_byte_share, bytes_per_pkt. hour/day NOT possible (no timestamps) (affected 0; rows 257,673 -> 257,673)
 - Keys & dtypes: orig_id kept (per-file id); new global flow_id; categoricals stored as category; numerics downcast (affected 0; rows 257,673 -> 257,673)


Saved flows_clean.parquet: (257673, 61), 46.4 MB


{'rows_before': 257673,
 'rows_after': 257673,
 'cols_after': 61,
 'memory_mb_before': np.float64(49.2),
 'memory_mb_after': np.float64(46.4),
 'duplicates_flagged': 93824,
 'duplicates_by_split': {'test': 26387, 'train': 67437},
 'test_rows_seen_in_train': 4247,
 'invalid_ftp_login': 26,
 'zero_duration': 3607,
 'label_mismatch': 0,
 'negatives': 0}

## Cleaning log

In [3]:
display(Markdown((REPORTS_DIR / 'cleaning_log.md').read_text(encoding='utf-8')))

# Cleaning log

Generated by `src/clean.py`. Dataset: UNSW-NB15 official train/test release.

| # | Step | Decision | Rows affected | Rows before | Rows after |
|---|------|----------|--------------:|------------:|-----------:|
| 1 | Column names | Header row present; names stripped and lower-cased (no NUSW feature-file mapping needed for this release) | 0 | 257,673 | 257,673 |
| 2 | Unavailable fields | srcip, dstip, sport, dsport, stime, ltime do NOT exist in the train/test release -> no epoch conversion, port coercion, hour/day features or IP analysis. Substitutes are documented in the README | 0 | 257,673 | 257,673 |
| 3 | Categorical text | Stripped whitespace; proto/service lower-case, state upper-case | 1 | 257,673 | 257,673 |
| 4 | Service '-' | '-' means no application service was identified -> recoded to 'none' | 141,321 | 257,673 | 257,673 |
| 5 | attack_cat missing | NaN/blank attack_cat filled with 'Normal' | 0 | 257,673 | 257,673 |
| 6 | attack_cat labels | Stripped + mapped spelling variants (e.g. 'Backdoors'->'Backdoor') | 0 | 257,673 | 257,673 |
| 7 | Label consistency | Checked label==0 <=> attack_cat=='Normal'; mismatches flagged (flag_label_mismatch) | 0 | 257,673 | 257,673 |
| 8 | Negative values | Scanned all numeric columns for negatives; flagged (flag_negative) | 0 | 257,673 | 257,673 |
| 9 | TTL range | TTL must be 0-255; out-of-range rows flagged | 0 | 257,673 | 257,673 |
| 10 | is_ftp_login | Binary feature had values 2-4; capped to 1 and flagged (flag_invalid_ftp_login) | 26 | 257,673 | 257,673 |
| 11 | Zero duration | dur==0 kept (single-packet flows) but flagged; rate features use dur clipped at 1 microsecond | 3,607 | 257,673 | 257,673 |
| 12 | Duplicates (within split) | Feature-identical rows (ignoring id) FLAGGED, not removed: dropping them would change the official benchmark split, and repeated identical flows are genuine traffic (e.g. scripted Generic/Exploit attacks) | 93,824 | 257,673 | 257,673 |
| 13 | Train/test overlap | Test rows whose features exactly match a train row flagged (in_both_splits) - reported as an evaluation caveat | 4,247 | 257,673 | 257,673 |
| 14 | Derived features | Added is_attack, total_bytes, total_pkts, bytes_per_sec, pkts_per_sec, byte_ratio=sbytes/(dbytes+1), src_byte_share, bytes_per_pkt. hour/day NOT possible (no timestamps) | 0 | 257,673 | 257,673 |
| 15 | Keys & dtypes | orig_id kept (per-file id); new global flow_id; categoricals stored as category; numerics downcast | 0 | 257,673 | 257,673 |

## Summary

- Rows before: 257,673; rows after: 257,673 (no rows dropped - problems are flagged so analysts can filter them).
- Memory: 49.2 MB raw (downcast) -> 46.4 MB cleaned (categoricals).
- Duplicates flagged: 93,824 {'test': 26387, 'train': 67437}.
- Test rows identical to a train row: 4,247.
- Output: `data/processed/flows_clean.parquet` (61 columns).


## Cleaned dataset
Saved as Parquet with categorical dtypes and derived features.

In [4]:
from config import CLEAN_PARQUET
df = pd.read_parquet(CLEAN_PARQUET)
print(df.shape)
df[['dur','total_bytes','total_pkts','bytes_per_sec','pkts_per_sec','byte_ratio','is_duplicate','in_both_splits']].describe().T

(257673, 61)


,count,mean,std,min,25%,50%,75%,max
dur,257673.0,1.246715e+00,5.974306e+00,0.000000,0.000008,0.004285,6.857770e-01,5.999999e+01
total_bytes,257673.0,2.296024e+04,2.282063e+05,24.000000,114.000000,832.000000,3.084000e+03,1.472823e+07
total_pkts,257673.0,3.829185e+01,2.066178e+02,1.000000,2.000000,8.000000,2.200000e+01,1.266000e+04
bytes_per_sec,257673.0,1.852400e+07,4.792043e+07,0.766777,5399.186035,770734.750000,2.222222e+07,1.768000e+09
pkts_per_sec,257673.0,1.973922e+05,3.459512e+05,0.016669,35.397255,4032.258057,2.500000e+05,2.000000e+07
byte_ratio,257673.0,2.143441e+02,1.508839e+03,0.001974,0.936170,21.007059,1.140000e+02,5.494400e+04
is_duplicate,257673.0,3.641204e-01,4.811836e-01,0.000000,0.000000,0.000000,1.000000e+00,1.000000e+00
in_both_splits,257673.0,1.648213e-02,1.273206e-01,0.000000,0.000000,0.000000,0.000000e+00,1.000000e+00
